# Functions

In [ ]:
import time


In [ ]:
# Optimised
def Yearly(startDate, endDate, frequency = 'YS'): #ReadDatesForFileFinding - Gets range of dates in given frequency to use to get the data files.
    YearlyRange = pd.date_range(startDate, endDate, freq=frequency).strftime("%Y%m%d").tolist()
    return YearlyRange

# Convert a dataframe to a series
def dfToSeries(DataFrame):
    DataFrame.insert(0, "DateTime", pd.to_datetime(DataFrame['Year'] * 10000000 + DataFrame['DOY']*10000 + DataFrame['Hour']*100 + DataFrame['Minute'], format='%Y%j%H%M'))
    DataFrameDateTimeIndex = DataFrame.set_index('DateTime') 
    SunpyTimeSeries =  GenericTimeSeries(DataFrameDateTimeIndex)
    return SunpyTimeSeries

# Get to date portion of the file names
def myFunc(e):
  return e.split("b")[1]

# Wind: Get the pre-downloaded data from folder
def FindDataWind(directory, dataName, startDate, endDate, frequency = 'MS'): #FindDataMakeTimeSeries
    files = []
    for file in os.listdir(directory):
        filename = os.fsdecode(file)
        DateRange = pd.date_range(startDate, endDate, freq=frequency).strftime("%Y%m%d").tolist()
        for j in range(len(DateRange)):
            if filename.startswith(dataName + str(DateRange[j])[0:6]):
                files.append(directory+filename)
                continue
    filesSorted = sorted(files, key=myFunc) # sort files
    df_list = [] #pd.DataFrame([])
    # Columns from https://omniweb.gsfc.nasa.gov/html/HROdocum.html
    colnames = ["Year", "DOY", "Hour", "Minute", "IMF_PTS", "PercentInterp", "CP/MVFlag", "Timeshift", "PFN_XGSE", "PFN_YGSE","PFN_ZGSE", "ScalarB", "Bx_GSEGSM", "ByGSE", "BzGSE", "ByGSM", "BzGSM", "RMSTimeshift", "RMS_PFN", "RMSScalarB", "RMSFieldVector", "Plasma_PTS", "FlowSpeed", "Vx_GSE", "Vy_GSE", "Vz_GSE", "ProtonDensity", "Temp", "X_s_c_GSE", "Y_s_c_GSE", "Z_s_c_GSE", "X_targ_GSE", "Y_targ_GSE", "Z_targ_GSE", "RMS_targ", "DBOT1", "DBOT2"]
    for j in range(len(filesSorted)):
        df_list.append( pd.read_csv(filesSorted[j], header=None, sep = "\s+", names = colnames) )
    dataTimeSeries = pd.concat(df_list, ignore_index=True)
    
    WindFillerVals = [999, 9.9, 999999, 99.99, 9999.99, 99999.9, 999.99, 9999999] #https://omniweb.gsfc.nasa.gov/html/HROdocum.html#4b
    dataTimeSeriesFillersToNans = dataTimeSeries.replace(WindFillerVals, np.nan) # replace filler values with nans
    return dataTimeSeriesFillersToNans

# Calculate Total Vsw, default values for Wind
def AddTotVsw(dataTimeSeries, VxColName = "Vx_GSE", VyColName = "Vy_GSE", VzColName = "Vz_GSE"):
    TotVsw = np.sqrt(dataTimeSeries.to_dataframe()[VxColName]**2 + 
                     dataTimeSeries.to_dataframe()[VyColName]**2 + 
                     dataTimeSeries.to_dataframe()[VzColName]**2)
    dataTimeSeries = dataTimeSeries.add_column("TotVsw", TotVsw)
    return dataTimeSeries

# Combine the components of magnetic field from data to get magnitude of magnetic field, BMag, default values for Wind
def AddBMag(dataTimeSeries, BxColName = "Bx_GSEGSM", ByColName = "ByGSE", BzColName = "BzGSE"):
    BMag = np.sqrt(dataTimeSeries.to_dataframe()[BxColName]**2 + 
                   dataTimeSeries.to_dataframe()[ByColName]**2 + 
                   dataTimeSeries.to_dataframe()[BzColName]**2)
    dataTimeSeries = dataTimeSeries.add_column("BMag", BMag)
    return dataTimeSeries

# Combine the components of magnetic field from data to get magnitude of magnetic field, BMag, default values for Wind
def AddLogBMag(dataTimeSeries):
    LogB = np.log(dataTimeSeries.to_dataframe()["BMag"])
    dataTimeSeries = dataTimeSeries.add_column("LogB", LogB)
    return dataTimeSeries

# Combine the components of magnetic field from data to get magnitude of magnetic field, BMag, default values for Wind
def AddLogBX(dataTimeSeries, BxColName = "Bx_GSEGSM"):
    LogBX = np.log(dataTimeSeries.to_dataframe()[BxColName])
    dataTimeSeries = dataTimeSeries.add_column("LogBX", LogBX)
    return dataTimeSeries

#Plot Vsw, default values for Wind
def PlotVsw(dataTimeSeries, Columns = ["TotVsw", "Vx_GSE", "Vy_GSE", "Vz_GSE"], YLim = [-800, 800], saving = False):
    # dataTimeSeries[cols] = dataTimeSeries[dataTimeSeries[cols] > filter][cols]
    fig, ax = plt.subplots()
    dataTimeSeries.plot(columns=Columns, alpha = 0.5)
    plt.legend(loc="upper right")
    plt.ylim(YLim[0], YLim[1])
    # print(dataTSFirstDates.to_dataframe()[""].unique())

    if saving == True:
        timestr = time.strftime("%Y%m%d-%H%M%S")
        plt.savefig(PlotDirectory + "PlotVsw" + timestr + ".png")
    else:
        print("File not saved")

#Plot B, default values for Wind
def PlotB(dataTimeSeries, Columns=["BMag", "Bx_GSEGSM", "ByGSE", "BzGSE"], YLim = [-20,20], saving = False):
    fig, ax = plt.subplots()
    dataTimeSeries.plot(columns=Columns, alpha = 0.5)
    plt.legend(loc="upper right")
    plt.ylim(YLim[0], YLim[1])
    # print(dataTSFirstDates.to_dataframe()[""].unique())

    if saving == True:
        timestr = time.strftime("%Y%m%d-%H%M%S")
        plt.savefig(PlotDirectory + "PlotB" + timestr + ".png")
    else:
        print("File not saved")

# Dotted plots
def PlotVswDotted(dataTimeSeries, Columns = ["TotVsw", "Vx_GSE", "Vy_GSE", "Vz_GSE"], Units = ["km/s", "km/s", "km/s", "km/s"], YLim = [-800, 800], saving = False):
    fig, ax  = plt.subplots(len(Columns), sharex=True, figsize=(10, 10))

    for i in range(len(Columns)):
        # total Wind speed
        dataTimeSeries.to_dataframe()[Columns[i]].plot(ax=ax[i], marker='.', ls='', ms=1)
        ax[i].set_ylabel(Columns[i] + " " + Units[i])
    # # flow speed
    # dataTimeSeries.to_dataframe()["FlowSpeed"].plot(ax=ax[1], marker='.', ls='', ms=1)
    # ax[1].set_ylabel("Flow speed (km/s)")
    # components of Wind speed, Vx
    # dataTimeSeries.to_dataframe()["Vx_GSE"].plot(ax=ax[1], marker='.', ls='', ms=1) # this is getting negative results, which makes sense if V_GSM_0 is the X GSM axis (towards the Sun) since the SW will be flowing in the opposite direction.
    # ax[1].set_ylabel("Vx V$_{SW}$ (km/s)")
    # # components of Wind speed, Vy
    # dataTimeSeries.to_dataframe()["Vy_GSE"].plot(ax=ax[2], marker='.', ls='', ms=1) # this is likely Y then (essentially along the west->east line of the (center of the) Earth).
    # ax[2].set_ylabel("Vy V$_{SW}$ (km/s)")
    # # components of Wind speed, Vz
    # dataTimeSeries.to_dataframe()["Vz_GSE"].plot(ax=ax[3], marker='.', ls='', ms=1) # and this is likely Z then (essentially along the south->north line of the (center of the) Earth).  I guess you'd expect these components to be lower than the radial one as the solar OMNI is mostly radial/along the Parker spiral
    # ax[3].set_ylabel("Vz V$_{SW}$ (km/s)")

    for aa in ax:
        aa.legend(loc="upper left")
        # aa.axvline("2022-01-29 00:00", color='k')
        # aa.axvline("2022-01-30 02:00", color='k', ls="dashed")
        aa.set_ylim(YLim[0], YLim[1])

    # ax[4].set_xlabel("Time")
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.02)

    if saving == True:
        timestr = time.strftime("%Y%m%d-%H%M%S")
        plt.savefig(PlotDirectory + "VswDotted" + timestr + ".png")
    else:
        print("File not saved")

#Count quantity of data, default values from Wind
def DataQuantity(DataSerieses, Columns = ["TotVsw", "BMag", "Bx_GSEGSM", "ByGSE", "BzGSE"]):
    noOfPoints = []
    for i in range(len(DataSerieses)):
        for j in range(len(Columns)):
            dataUsed = DataSerieses[i].to_dataframe()[Columns[j]]
            dataUsed = dataUsed[np.isfinite(dataUsed)] # remove nans
            # print("DataSeries Year " + str(i+1) + "  = " + str(len(dataUsed))) # Print number of values for each histogram
            noOfPoints.append(len(dataUsed))
    print("Maximum no. of points = " + str(max(noOfPoints)))
    print("Minimum no. of points = " + str(min(noOfPoints)))
    print("Average no. of points = " + str(np.average(noOfPoints)))

#Plot histograms with a split - sharing y axis that checks across all years
def CycleOverlaps(dataTS1, dataTS2, dataTS3, FigSize, startSplit1, endSplit1, startSplit2, endSplit2, Bins = [50, 50, 50, 50, 50], LegLoc = 'upper right', TitleLoc = 0.9, SpaceLR = [100, 5, 5, 5, 5], VarsToPlot = ["TotVsw", "BMag", "Bx_GSEGSM", "ByGSE", "BzGSE"], Units = ["km/s", "nT", "nT", "nT", "nT"], XAxisScale = ["linear", "log", "linear", "linear", "linear"], saving = False, noOfYears = 11):

    maxYOverall = []
    yScaleLim = []
    XScaleMaxLim = []
    XScaleMinLim = []

    meansTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS1[:] = np.nan
    mediansTS1[:] = np.nan
    stdDevsTS1[:] = np.nan
    meansTS2[:] = np.nan
    mediansTS2[:] = np.nan
    stdDevsTS2[:] = np.nan
    meansTS3[:] = np.nan
    mediansTS3[:] = np.nan
    stdDevsTS3[:] = np.nan
    
    for i in range(len(VarsToPlot)):

        fig, axs = plt.subplots( (endSplit1-startSplit1) , 1, figsize=(FigSize[0], FigSize[1]*(endSplit1-startSplit1)/11), sharex=True)
        fig.subplots_adjust(hspace=0)
        plt.suptitle("Wind Data, " + VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=TitleLoc)
        fig.supxlabel(VarsToPlot[i] + ", " + Units[i], y=0.085)
        plt.xscale(XAxisScale[i])
        # subplotLabels = []

        maxYInDf11 = 0
        maxYInDf21 = 0
        maxYInDf31 = 0
        maxYInDf12 = 0
        maxYInDf22 = 0
        maxYInDf32 = 0

        maxXInDf11 = 0
        maxXInDf21 = 0
        maxXInDf31 = 0
        maxXInDf12 = 0
        maxXInDf22 = 0
        maxXInDf32 = 0

        minXInDf11 = 0
        minXInDf21 = 0
        minXInDf31 = 0
        minXInDf12 = 0
        minXInDf22 = 0
        minXInDf32 = 0

        # for j in range(len(dataTS1)):
        for j in range(startSplit1, endSplit1):

            # axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'orange', label = "Cycle 23", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

            y, x, _ = axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'orange', label = "Cycle 23", density = True)
            maxInYear = y.max()
            if maxInYear > maxYInDf11:
                maxYInDf11 = maxInYear

            maxXInYear = x.max()
            if maxXInYear > maxXInDf11:
                maxXInDf11 = maxXInYear

            minXInYear = x.min()
            if minXInYear < minXInDf11:
                minXInDf11 = minXInYear

            # C23
            #Mean
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "orange", label="mean = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
            #Median
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "orange", label="median = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(),2)))
            #Standard deviation
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

            meansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].mean()
            mediansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].median()
            stdDevsTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].std()

            if j < len(dataTS2):
                # axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'green', label = "Cycle 24", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf21:
                    maxYInDf21 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf21:
                    maxXInDf21 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf21:
                    minXInDf21 = minXInYear

                # C24
                #Mean
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "green", label="mean = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "green", label="median = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].std()

            if j < len(dataTS3):
                # axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'blue', label = "Cycle 25", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf31:
                    maxYInDf31 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf31:
                    maxXInDf31 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf31:
                    minXInDf31 = minXInYear

                # C25
                #Mean
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "blue", label="mean = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "blue", label="median = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].std()

        # for j in range(len(NoOfYears)):
        #     if j < len(dataTS):
            axs[j].legend(loc = LegLoc, fontsize = Legend)
            axs[j].set_ylabel("Normalised frequency")

            # subplotLabels.append("Year" + str(j))
            axs[j].annotate("Year " + str(j + 1), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))
            # axs[j].annotate(str(StartAndEnd[j][0][0:4]), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

        fig2, axs2 = plt.subplots( (endSplit2-startSplit2) , 1, figsize=(FigSize[0], FigSize[1]*(endSplit2-startSplit2)/11), sharex=True)
        fig2.subplots_adjust(hspace=0)
        plt.suptitle("Wind Data, " + VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.9)
        fig2.supxlabel(VarsToPlot[i] + ", " + Units[i], y=0.085)
        plt.xscale(XAxisScale[i])
        # subplotLabels = []

        for j in range(startSplit2, endSplit2):
            axNo = j-startSplit2

            y, x, _ = axs2[axNo].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'orange', label = "Cycle 23", density = True)
            maxInYear = y.max()
            if maxInYear > maxYInDf12:
                maxYInDf12 = maxInYear

            maxXInYear = x.max()
            if maxXInYear > maxXInDf12:
                maxXInDf12 = maxXInYear

            minXInYear = x.min()
            if minXInYear < minXInDf12:
                minXInDf12 = minXInYear

            # C23
            #Mean
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "orange", label="mean = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
            #Median
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "orange", label="median = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(),2)))
            #Standard deviation
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

            meansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].mean()
            mediansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].median()
            stdDevsTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].std()
            
            if j < len(dataTS2):
                # axs[axNo].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs2[axNo].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'green', label = "Cycle 24", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf22:
                    maxYInDf22 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf22:
                    maxXInDf22 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf22:
                    minXInDf22 = minXInYear

                # C24
                #Mean
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "green", label="mean = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "green", label="median = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].std()

            if j < len(dataTS3):
                # axs[axNo].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs2[axNo].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'blue', label = "Cycle 25", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf32:
                    maxYInDf32 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf32:
                    maxXInDf32 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf32:
                    minXInDf32 = minXInYear

                # C25
                #Mean
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "blue", label="mean = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "blue", label="median = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].std()
                
        # for j in range(len(NoOfYears)):
        #     if j < len(dataTS):
            axs2[axNo].legend(loc = LegLoc, fontsize = Legend)
            axs2[axNo].set_ylabel("Normalised frequency")

            # subplotLabels.append("Year" + str(j))
            axs2[axNo].annotate("Year " + str(j+1), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))
            # axs2[axNo].annotate(str(StartAndEnd[j][0][0:4]), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

        maxYsInDfs = [maxYInDf11, maxYInDf21, maxYInDf31, maxYInDf12, maxYInDf22, maxYInDf32]
        maxYOverall = max(maxYsInDfs)
        yScaleLim.append(maxYOverall)

        for j in range(startSplit1, endSplit1):
            # axs[j].set_ylim(0, maxYOverall)
            axs[j].set_ylim(0, maxYOverall*1.1) # adding 10% to y axis lim


        for j in range(startSplit2, endSplit2):
            axNo = j-startSplit2
            # axs2[axNo].set_ylim(0, maxYOverall)
            axs2[axNo].set_ylim(0, maxYOverall*1.1) # adding 10% to y axis lim

        maxXsInDfs = [maxXInDf11, maxXInDf21, maxXInDf31, maxXInDf12, maxXInDf22, maxXInDf32]
        maxXOverall = max(maxXsInDfs)
        XScaleMaxLim.append(maxXOverall)

        minXsInDfs = [minXInDf11, minXInDf21, minXInDf31, minXInDf12, minXInDf22, minXInDf32]
        minXOverall = min(minXsInDfs)
        XScaleMinLim.append(minXOverall)

        if saving == True:
            timestr = time.strftime("%Y%m%d-%H%M%S")
            fig.savefig(PlotDirectory + StartAndEnd[startSplit1][0][0:10] + "->" + StartAndEnd[endSplit1][1][0:10] + VarsToPlot[i] + "FirstHalf_Wind" + timestr + ".png")
            fig2.savefig(PlotDirectory + StartAndEnd[startSplit2][0][0:10] + "->" + StartAndEnd[endSplit2][1][0:10] + VarsToPlot[i] + "SecondHalf_Wind" + timestr + ".png")

        else:
            print("File not saved")

        plt.show()
    return yScaleLim, XScaleMaxLim, XScaleMinLim, meansTS1, meansTS2, meansTS3, mediansTS1, mediansTS2, mediansTS3, stdDevsTS1, stdDevsTS2, stdDevsTS3




In [ ]:
# # Get the pre-downloaded data from folder - Started to try and make the code easy to switch between wind/omni but later code will face issues as colnames are different so left for now
# def FindData(DataName, startDate, endDate): #FindDataMakeTimeSeries
#     if DataName == "Wind":
#         directory = "Data/windData/"
#         dataName = "wind_min_b" # b = shifted to bow shock nose (by technique 4)
#         files = []
#         for file in os.listdir(directory):
#             filename = os.fsdecode(file)
#             DateRange = ReadDates(startDate, endDate)
#             for j in range(len(DateRange)):
#                 if filename.startswith(dataName + str(DateRange[j])[0:6]):
#                     files.append(directory+filename)
#                     continue
#         filesSorted = sorted(files, key=myFunc) # sort files
#         df_list = [] #pd.DataFrame([])
#         # Columns from https://omniweb.gsfc.nasa.gov/html/HROdocum.html
#         colnames = ["Year", "DOY", "Hour", "Minute", "IMF_PTS", "PercentInterp", "CP/MVFlag", "Timeshift", "PFN_XGSE", "PFN_YGSE","PFN_ZGSE", "ScalarB", "Bx_GSEGSM", "ByGSE", "BzGSE", "ByGSM", "BzGSM", "RMSTimeshift", "RMS_PFN", "RMSScalarB", "RMSFieldVector", "Plasma_PTS", "FlowSpeed", "Vx_GSE", "Vy_GSE", "Vz_GSE", "ProtonDensity", "Temp", "X_s_c_GSE", "Y_s_c_GSE", "Z_s_c_GSE", "X_targ_GSE", "Y_targ_GSE", "Z_targ_GSE", "RMS_targ", "DBOT1", "DBOT2"]
#         for j in range(len(filesSorted)):
#             df_list.append( pd.read_csv(filesSorted[j], header=None, sep = "\s+", names = colnames) )
#         dataTimeSeries = pd.concat(df_list, ignore_index=True)

#         # replace with nans
#         WindFillerVals = [999, 9.9, 999999, 99.99, 9999.99, 99999.9, 999.99, 9999999] #https://omniweb.gsfc.nasa.gov/html/HROdocum.html#4b
#         dataTimeSeriesFillersToNans = dataTimeSeries.replace(WindFillerVals, np.nan)
#     elif DataName == "Omni":
#         directory = "./Data/omniData/"
#         dataName = "omni_hro2_1min_"
#         files = []
#         for file in os.listdir(directory):
#             filename = os.fsdecode(file)
#             DateRange = ReadDates(startDate, endDate)
#             for j in range(len(DateRange)):
#                 if filename.startswith(dataName + str(DateRange[j])):
#                     files.append(directory+filename)
#                     continue
#         filesSorted = sorted(files, key=myFunc) # sort files
#         dataTimeSeries = TimeSeries(filesSorted, concatenate=True)

#         # replace with nans
#         OmniFillerVals = [99, 999, 999999, 99.99, 9999.99, 99999.9, 999.99, 999.9, 99999, 9.999, 99.9, 99999.99] #https://omniweb.gsfc.nasa.gov/html/HROdocum.html#4b
#         dataTimeSeriesFillersToNans = dataTimeSeries.replace(OmniFillerVals, np.nan)
#     else: 
#         print("Named data not available.")

#     return dataTimeSeriesFillersToNans


# # Plot histograms all together
# def PlotHistTogether(dataTimeSerieses, VarsToPlot):
#     for i in range(len(VarsToPlot)):
#         maxY = 0
#         for j in range(len(dataTimeSerieses)):
#             plt.hist(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]], bins = Bin, label = "Wind " + str(StartAndEnd[j][0] + " - " + str(StartAndEnd[j][1])), alpha = Alpha, density = True)

#             y, x, _ = plt.hist(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]], bins = Bin, label = "Wind " + str(StartAndEnd[j][0][0:10] + " - " + str(StartAndEnd[j][1][0:10])), alpha = Alpha, density = True)
#             maxVarY = y.max()
#             if maxVarY > maxY:
#                 maxY = maxVarY
#             # print(maxVarY, maxY)

#         plt.title("Solar Wind " + VarsToPlot[i])
#         plt.xlabel(VarsToPlot[i] + r", GSE, $km/s$")
#         plt.ylabel("Normalised frequency")
#         plt.legend(fontsize=Legend)
#         # plt.ylim(0, maxY)
#         plt.ylim(0, maxY+ 0.1*maxY)
#         plt.show()

# #Plot histograms separately
# def PlotHistSeparate(dataTimeSerieses, VarsToPlot, XAxisScale, YAxisScale, FigSize):
#     for i in range(len(VarsToPlot)): # For each variable
#         # LowerBounds = [] # set lower bounds array to 0
#         # UpperBounds = [] # set upper bounds array to 0
#         fig, axs = plt.subplots(len(dataTimeSerieses),1, figsize=FigSize, sharex=True)
#         fig.subplots_adjust(hspace=0)
#         plt.suptitle("Solar Wind "+ VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.89)
#         fig.supxlabel(VarsToPlot[i] + ", " + Units[i], y=0.099)

#         maxY = 0
#         for j in range(len(dataTimeSerieses)): # For each daterange
#             # axs[j].hist(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]], bins = Bin, label = "Wind " + str(StartAndEnd[j][0]) + " - " + str(StartAndEnd[j][1]), color=Colours[j], alpha = Alpha, density = True)

#             y, x, _ = axs[j].hist(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]], bins = Bin, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True)
#             maxVarY = y.max()
#             if maxVarY > maxY:
#                 maxY = maxVarY
#             # print(maxVarY, maxY)


#             #Mean
#             axs[j].axvline(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "grey", label="mean = " + str(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].mean()))
#             #Median
#             axs[j].axvline(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = '--', color = "black", label="median = " + str(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].median()))
#             #Standard deviation
#             axs[j].axvline(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].std()), alpha = 0)

#             axs[j].legend(loc = LegLoc, fontsize = Legend)
#                 # axs[1].legend(loc = LegLoc, fontsize = Legend)
#                 # axs[2].legend(loc = LegLoc, fontsize = Legend)
#                 # axs[3].legend(loc = LegLoc, fontsize = Legend)
            
#             # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].values))
#             # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].values))
#             # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             axs[j].set_ylabel("Normalised frequency")
        

#         # plt.xlim(min(LowerBounds) - SpaceLR[i], max(UpperBounds) + SpaceLR[i])
#         # plt.xlim(np.nanmin(LowerBounds) - SpaceLR[i], np.nanmax(UpperBounds) + SpaceLR[i])

#         plt.xscale(XAxisScale[i])
        

#         # for m in range(len(dataTimeSerieses)):
#         #     if maxY == np.nan or maxY == np.inf:
#         #             print("Max is nan or inf.")
#             # else:
#         for m in range(len(dataTimeSerieses)):
#             # axs[m].set_ylim(0, maxY)
#             axs[m].set_ylim(0, maxY+ 0.1*maxY)
#         # plt.tight_layout()
#         plt.yscale(YAxisScale[i])
        
#         plt.savefig(PlotDirectory + StartAndEnd[0][0][0:10] + "->" + StartAndEnd[0][0][0:10] + VarsToPlot[i] + "_Wind.png")
#         # plt.show()


# #Plot histograms separately - no split
# def YearOverlap(dataTS1, dataTS2, dataTS3, VarsToPlot, NoOfYears, FigSize, TitleLoc, startYear): #, startSplit, endSplit):
    
#     for i in range(len(VarsToPlot)):
#         fig, axs = plt.subplots(NoOfYears,1, figsize=FigSize, sharex=True)
#         fig.subplots_adjust(hspace=0)
#         # plt.suptitle("Solar Wind "+ VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.89)
#         fig.supxlabel("Wind Data" + VarsToPlot[i] + ", " + Units[i], y=TitleLoc)
#         # subplotLabels = []

#         maxYInDf1 = 0
#         maxYInDf2 = 0
#         maxYInDf3 = 0
#         for j in range(len(dataTS1)):
#         # for j in range(startSplit, endSplit):

#             # axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'orange', label = "Cycle 23", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

#             y, x, _ = axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'orange', label = "Cycle 23", density = True)
#             maxInYear = y.max()
#             if maxInYear > maxYInDf1:
#                 maxYInDf1 = maxInYear

#             # C23
#             #Mean
#             axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "orange", label="mean = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
#             #Median
#             axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "orange", label="median = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(),2)))
#             #Standard deviation
#             axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)


#             if j < len(dataTS2):
#                 axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

#                 y, x, _ = axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True)
#                 maxInYear = y.max()
#                 if maxInYear > maxYInDf2:
#                     maxYInDf2 = maxInYear

#                 # C24
#                 #Mean
#                 axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "green", label="mean = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
#                 #Median
#                 axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "green", label="median = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
#                 #Standard deviation
#                 axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)


#             if j < len(dataTS3):
#                 axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

#                 y, x, _ = axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True)
#                 maxInYear = y.max()
#                 if maxInYear > maxYInDf3:
#                     maxYInDf3 = maxInYear

#                 # C25
#                 #Mean
#                 axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "blue", label="mean = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
#                 #Median
#                 axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "blue", label="median = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
#                 #Standard deviation
#                 axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

#         # for j in range(len(NoOfYears)):
#         #     if j < len(dataTS):
#             axs[j].legend(loc = LegLoc, fontsize = Legend)
#             axs[j].set_ylabel("Normalised frequency")

#             # subplotLabels.append("Year" + str(j))
#             axs[j].annotate("Year " + str(startYear + j), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))


#             # fig, axs = plt.subplot_mosaic([['a)', 'c)'], ['b)', 'c)'], ['d)', 'd)']], layout='constrained')
#             # for label, ax in axs.items():
#             #     ax.annotate(label, xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

#         # maxYInDf1 = 0
#         # for k in range(len(dataTS1)): # For each daterange
#         #     maxInYear = max(dataTS1[k].to_dataframe()[VarsToPlot[i]])

#         #     if maxInYear > maxYInDf1:
#         #         maxYInDf1 = maxInYear

#         # maxYInDf2 = 0
#         # for l in range(len(dataTS2)): # For each daterange
#         #     maxInYear = max(dataTS2[l].to_dataframe()[VarsToPlot[i]])
            
#         #     if maxInYear > maxYInDf2:
#         #         maxYInDf2 = maxInYear

#         # maxYInDf3 = 0
#         # for m in range(len(dataTS3)): # For each daterange
#         #     maxInYear = max(dataTS3[m].to_dataframe()[VarsToPlot[i]])
            
#         #     if maxInYear > maxYInDf3:
#         #         maxYInDf3 = maxInYear
            
#         maxYsInDfs = [maxYInDf1, maxYInDf2, maxYInDf3]
#         maxYOverall = max(maxYsInDfs)
#         for j in range(len(dataTS1)):
#             axs[j].set_ylim(0, maxYOverall)

            
#             #     # axs[1].legend(loc = LegLoc, fontsize = Legend)
#             #     # axs[2].legend(loc = LegLoc, fontsize = Legend)
#             #     # axs[3].legend(loc = LegLoc, fontsize = Legend)
            
#             # # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].values))
#             # # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]].values))
#             # # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # # LowerBounds.append(min(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # # UpperBounds.append(max(dataTimeSerieses[j].to_dataframe()[Plotting[i]].values))
#             # axs[j].set_ylabel("Normalised frequency")
        

#         # # plt.xlim(min(LowerBounds) - SpaceLR[i], max(UpperBounds) + SpaceLR[i])
#         # # plt.xlim(np.nanmin(LowerBounds) - SpaceLR[i], np.nanmax(UpperBounds) + SpaceLR[i])

#         # plt.xscale(XAxisScale[i])
        

#         # # for m in range(len(dataTimeSerieses)):
#         # #     if maxY == np.nan or maxY == np.inf:
#         # #             print("Max is nan or inf.")
#         #     # else:
#         # for m in range(len(dataTimeSerieses)):
#         #     # axs[m].set_ylim(0, maxY)
#         #     axs[m].set_ylim(0, maxY+ 0.1*maxY)
#         # # plt.tight_layout()
#         # plt.yscale(YAxisScale[i])
        
#         # plt.savefig("Plots/Wind/" + StartAndEnd[0][0][0:10] + "->" + StartAndEnd[0][0][0:10] + VarsToPlot[i] + "_Wind.png")
#         plt.show()


#Plot histograms with a split - sharing y axis that checks across all years
def YearOverlapGlobalYAxis(dataTS1, dataTS2, dataTS3, VarsToPlot, FigSize, TitleLoc, startSplit1, endSplit1, startSplit2, endSplit2, XAxisScale, Bins):

    maxYOverall = []
    yScaleLim = []
    XScaleMaxLim = []
    XScaleMinLim = []

    meansTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS1[:] = np.nan
    mediansTS1[:] = np.nan
    stdDevsTS1[:] = np.nan
    meansTS2[:] = np.nan
    mediansTS2[:] = np.nan
    stdDevsTS2[:] = np.nan
    meansTS3[:] = np.nan
    mediansTS3[:] = np.nan
    stdDevsTS3[:] = np.nan
    
    for i in range(len(VarsToPlot)):

        fig, axs = plt.subplots( (endSplit1-startSplit1) , 1, figsize=(FigSize[0], FigSize[1]*(endSplit1-startSplit1)/11), sharex=True)
        fig.subplots_adjust(hspace=0)
        plt.suptitle("Wind Data, " + VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=TitleLoc)
        fig.supxlabel(VarsToPlot[i] + ", " + Units[i], y=0.085)
        plt.xscale(XAxisScale[i])
        # subplotLabels = []

        maxYInDf11 = 0
        maxYInDf21 = 0
        maxYInDf31 = 0
        maxYInDf12 = 0
        maxYInDf22 = 0
        maxYInDf32 = 0

        maxXInDf11 = 0
        maxXInDf21 = 0
        maxXInDf31 = 0
        maxXInDf12 = 0
        maxXInDf22 = 0
        maxXInDf32 = 0

        minXInDf11 = 0
        minXInDf21 = 0
        minXInDf31 = 0
        minXInDf12 = 0
        minXInDf22 = 0
        minXInDf32 = 0

        # for j in range(len(dataTS1)):
        for j in range(startSplit1, endSplit1):

            # axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'orange', label = "Cycle 23", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

            y, x, _ = axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'orange', label = "Cycle 23", density = True)
            maxInYear = y.max()
            if maxInYear > maxYInDf11:
                maxYInDf11 = maxInYear

            maxXInYear = x.max()
            if maxXInYear > maxXInDf11:
                maxXInDf11 = maxXInYear

            minXInYear = x.min()
            if minXInYear < minXInDf11:
                minXInDf11 = minXInYear

            # C23
            #Mean
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "orange", label="mean = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
            #Median
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "orange", label="median = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(),2)))
            #Standard deviation
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

            meansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].mean()
            mediansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].median()
            stdDevsTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].std()

            if j < len(dataTS2):
                # axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'green', label = "Cycle 24", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf21:
                    maxYInDf21 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf21:
                    maxXInDf21 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf21:
                    minXInDf21 = minXInYear

                # C24
                #Mean
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "green", label="mean = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "green", label="median = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].std()

            if j < len(dataTS3):
                # axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'blue', label = "Cycle 25", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf31:
                    maxYInDf31 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf31:
                    maxXInDf31 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf31:
                    minXInDf31 = minXInYear

                # C25
                #Mean
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "blue", label="mean = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "blue", label="median = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].std()

        # for j in range(len(NoOfYears)):
        #     if j < len(dataTS):
            axs[j].legend(loc = LegLoc, fontsize = Legend)
            axs[j].set_ylabel("Normalised frequency")

            # subplotLabels.append("Year" + str(j))
            axs[j].annotate("Year " + str(j + 1), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))
            # axs[j].annotate(str(StartAndEnd[j][0][0:4]), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

            # fig, axs = plt.subplot_mosaic([['a)', 'c)'], ['b)', 'c)'], ['d)', 'd)']], layout='constrained')
            # for label, ax in axs.items():
            #     ax.annotate(label, xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

        # maxYInDf1 = 0
        # for k in range(len(dataTS1)): # For each daterange
        #     maxInYear = max(dataTS1[k].to_dataframe()[VarsToPlot[i]])

        #     if maxInYear > maxYInDf1:
        #         maxYInDf1 = maxInYear

        # maxYInDf2 = 0
        # for l in range(len(dataTS2)): # For each daterange
        #     maxInYear = max(dataTS2[l].to_dataframe()[VarsToPlot[i]])
            
        #     if maxInYear > maxYInDf2:
        #         maxYInDf2 = maxInYear

        # maxYInDf3 = 0
        # for m in range(len(dataTS3)): # For each daterange
        #     maxInYear = max(dataTS3[m].to_dataframe()[VarsToPlot[i]])
            
        #     if maxInYear > maxYInDf3:
        #         maxYInDf3 = maxInYear
            
        
        fig2, axs2 = plt.subplots( (endSplit2-startSplit2) , 1, figsize=(FigSize[0], FigSize[1]*(endSplit2-startSplit2)/11), sharex=True)
        fig2.subplots_adjust(hspace=0)
        plt.suptitle("Wind Data, " + VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.9)
        fig2.supxlabel(VarsToPlot[i] + ", " + Units[i], y=0.085)
        plt.xscale(XAxisScale[i])
        # subplotLabels = []

        for j in range(startSplit2, endSplit2):

            # fig, axs = plt.subplots( (endSplit2-startSplit2) , 1, figsize=FigSize, sharex=True)
            # fig.subplots_adjust(hspace=0)
            # # plt.suptitle("Solar Wind "+ VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.89)
            # fig.supxlabel("Wind Data" + VarsToPlot[i] + ", " + Units[i], y=TitleLoc)
            # # subplotLabels = []

            axNo = j-startSplit2

            # axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'orange', label = "Cycle 23", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

            y, x, _ = axs2[axNo].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'orange', label = "Cycle 23", density = True)
            maxInYear = y.max()
            if maxInYear > maxYInDf12:
                maxYInDf12 = maxInYear

            maxXInYear = x.max()
            if maxXInYear > maxXInDf12:
                maxXInDf12 = maxXInYear

            minXInYear = x.min()
            if minXInYear < minXInDf12:
                minXInDf12 = minXInYear

            # C23
            #Mean
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "orange", label="mean = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
            #Median
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "orange", label="median = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(),2)))
            #Standard deviation
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

            meansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].mean()
            mediansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].median()
            stdDevsTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].std()
            
            if j < len(dataTS2):
                # axs[axNo].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs2[axNo].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'green', label = "Cycle 24", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf22:
                    maxYInDf22 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf22:
                    maxXInDf22 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf22:
                    minXInDf22 = minXInYear

                # C24
                #Mean
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "green", label="mean = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "green", label="median = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].std()

            if j < len(dataTS3):
                # axs[axNo].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs2[axNo].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'blue', label = "Cycle 25", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf32:
                    maxYInDf32 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf32:
                    maxXInDf32 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf32:
                    minXInDf32 = minXInYear

                # C25
                #Mean
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "blue", label="mean = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "blue", label="median = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].std()
                
        # for j in range(len(NoOfYears)):
        #     if j < len(dataTS):
            axs2[axNo].legend(loc = LegLoc, fontsize = Legend)
            axs2[axNo].set_ylabel("Normalised frequency")

            # subplotLabels.append("Year" + str(j))
            axs2[axNo].annotate("Year " + str(j+1), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))
            # axs2[axNo].annotate(str(StartAndEnd[j][0][0:4]), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

        maxYsInDfs = [maxYInDf11, maxYInDf21, maxYInDf31, maxYInDf12, maxYInDf22, maxYInDf32]
        maxYOverall = max(maxYsInDfs)
        yScaleLim.append(maxYOverall)

        for j in range(startSplit1, endSplit1):
            # axs[j].set_ylim(0, maxYOverall)
            axs[j].set_ylim(0, maxYOverall*1.1) # adding 10% to y axis lim


        for j in range(startSplit2, endSplit2):
            axNo = j-startSplit2
            # axs2[axNo].set_ylim(0, maxYOverall)
            axs2[axNo].set_ylim(0, maxYOverall*1.1) # adding 10% to y axis lim




        maxXsInDfs = [maxXInDf11, maxXInDf21, maxXInDf31, maxXInDf12, maxXInDf22, maxXInDf32]
        maxXOverall = max(maxXsInDfs)
        XScaleMaxLim.append(maxXOverall)

        minXsInDfs = [minXInDf11, minXInDf21, minXInDf31, minXInDf12, minXInDf22, minXInDf32]
        minXOverall = min(minXsInDfs)
        XScaleMinLim.append(minXOverall)

        # # plt.xlim(min(LowerBounds) - SpaceLR[i], max(UpperBounds) + SpaceLR[i])
        # # plt.xlim(np.nanmin(LowerBounds) - SpaceLR[i], np.nanmax(UpperBounds) + SpaceLR[i])

        
        

        # # for m in range(len(dataTimeSerieses)):
        # #     if maxY == np.nan or maxY == np.inf:
        # #             print("Max is nan or inf.")
        #     # else:
        # for m in range(len(dataTimeSerieses)):
        #     # axs[m].set_ylim(0, maxY)
        #     axs[m].set_ylim(0, maxY+ 0.1*maxY)
        # # plt.tight_layout()
        # plt.yscale(YAxisScale[i])
        # fig.xscale(XAxisScale[i])
        # fig2.xscale(XAxisScale[i])
        
        fig.savefig(PlotDirectory + StartAndEnd[startSplit1][0][0:10] + "->" + StartAndEnd[endSplit1][1][0:10] + VarsToPlot[i] + "FirstHalf_Wind.png")
        fig2.savefig(PlotDirectory + StartAndEnd[startSplit2][0][0:10] + "->" + StartAndEnd[endSplit2][1][0:10] + VarsToPlot[i] + "SecondHalf_Wind.png")

        plt.show()
    return yScaleLim, XScaleMaxLim, XScaleMinLim, meansTS1, meansTS2, meansTS3, mediansTS1, mediansTS2, mediansTS3, stdDevsTS1, stdDevsTS2, stdDevsTS3


def StatsPlots(YearsArray, MeansTS1, MeansTS2, MeansTS3, MediansTS1, MediansTS2, MediansTS3, StdDevsTS1, StdDevsTS2, StdDevsTS3, FigSize, VarsToPlot, Units, xScaleMinLim, xScaleMaxLim, ManualYLims):
    # plt.suptitle("Solar Wind "+ VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.89)
    # subplotLabels = []
    for i in range(len(VarsToPlot)):

        fig, axs = plt.subplots(3,1, figsize=FigSize)#, sharex=True)
        fig.subplots_adjust(hspace=0)
        fig.supxlabel("Wind Data " + VarsToPlot[i] + ", " + Units[i], y=0.9)

        axs[0].plot(YearsArray, MeansTS1[i], label = "Cycle 23", color = "orange", marker = "o")
        axs[0].plot(YearsArray, MeansTS2[i], label = "Cycle 24", color = "green", marker = "o")
        axs[0].plot(YearsArray, MeansTS3[i], label = "Cycle 25", color = "blue", marker = "o")
        axs[0].set_ylabel("Mean")

        axs[1].plot(YearsArray, MediansTS1[i], label = "Cycle 23", color = "orange", marker = "o")
        axs[1].plot(YearsArray, MediansTS2[i], label = "Cycle 24", color = "green", marker = "o")
        axs[1].plot(YearsArray, MediansTS3[i], label = "Cycle 25", color = "blue", marker = "o")
        axs[1].set_ylabel("Median")

        axs[2].plot(YearsArray, StdDevsTS1[i], label = "Cycle 23", color = "orange", marker = "o")
        axs[2].plot(YearsArray, StdDevsTS2[i], label = "Cycle 24", color = "green", marker = "o")
        axs[2].plot(YearsArray, StdDevsTS3[i], label = "Cycle 25", color = "blue", marker = "o")
        axs[2].set_ylabel("Standard Deviation")
        axs[0].legend()
        # plt.title(VarsToPlot[i], Uni, y =3)
        plt.xlabel("Year of cycle")
        plt.xticks(np.arange(0, 11, 1), np.arange(1, 12, 1))

        # axs[0].set_ylim(xScaleMinLim[i], xScaleMaxLim[i]) # using original limits which include v high and low values, not great for seeing mean/median/std which tend to be mid values not extremes
        # axs[1].set_ylim(xScaleMinLim[i], xScaleMaxLim[i])
        # axs[2].set_ylim(xScaleMinLim[i], xScaleMaxLim[i])
        axs[0].set_ylim(ManualYLims[i][0]) # Manually chosen values
        axs[1].set_ylim(ManualYLims[i][1])
        axs[2].set_ylim(ManualYLims[i][2])

        # plt.savefig("Plots/WindYearlyPlots/" + VarsToPlot[i] + "StatPlots.png")
        plt.savefig(PlotDirectory + VarsToPlot[i] + "StatPlots.png")



#Plot histograms with a split - sharing y axis that checks across all years
def YearOverlapGlobalYAxisWithGaussianFitting(dataTS1, dataTS2, dataTS3, VarsToPlot, FigSize, TitleLoc, startSplit1, endSplit1, startSplit2, endSplit2, XAxisScale, Bins, FittingBool):

    maxYOverall = []
    yScaleLim = []
    XScaleMaxLim = []
    XScaleMinLim = []

    meansTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    mediansTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    stdDevsTS3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))

    meansTS1[:] = np.nan
    mediansTS1[:] = np.nan
    stdDevsTS1[:] = np.nan
    meansTS2[:] = np.nan
    mediansTS2[:] = np.nan
    stdDevsTS2[:] = np.nan
    meansTS3[:] = np.nan
    mediansTS3[:] = np.nan
    stdDevsTS3[:] = np.nan

    statsResult1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    statsResult1[:] = np.nan
    statsResult2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    statsResult2[:] = np.nan
    statsResult3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    statsResult3[:] = np.nan


    # critValues = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    # critValues[:] = np.nan

    pValues1 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    pValues1[:] = np.nan
    pValues2 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    pValues2[:] = np.nan
    pValues3 = np.empty((len(VarsToPlot),(endSplit2-startSplit1)))
    pValues3[:] = np.nan
    
    for i in range(len(VarsToPlot)):

        fig, axs = plt.subplots( (endSplit1-startSplit1) , 1, figsize=(FigSize[0], FigSize[1]*(endSplit1-startSplit1)/11), sharex=True)
        fig.subplots_adjust(hspace=0)
        plt.suptitle("Wind Data, " + VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=TitleLoc)
        fig.supxlabel(VarsToPlot[i] + ", " + Units[i], y=0.085)
        plt.xscale(XAxisScale[i])
        # subplotLabels = []

        maxYInDf11 = 0
        maxYInDf21 = 0
        maxYInDf31 = 0
        maxYInDf12 = 0
        maxYInDf22 = 0
        maxYInDf32 = 0

        maxXInDf11 = 0
        maxXInDf21 = 0
        maxXInDf31 = 0
        maxXInDf12 = 0
        maxXInDf22 = 0
        maxXInDf32 = 0

        minXInDf11 = 0
        minXInDf21 = 0
        minXInDf31 = 0
        minXInDf12 = 0
        minXInDf22 = 0
        minXInDf32 = 0

        # for j in range(len(dataTS1)):
        for j in range(startSplit1, endSplit1):

            # axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'orange', label = "Cycle 23", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

            y, x, _ = axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'orange', label = "Cycle 23", density = True)
            maxInYear = y.max()
            if maxInYear > maxYInDf11:
                maxYInDf11 = maxInYear

            maxXInYear = x.max()
            if maxXInYear > maxXInDf11:
                maxXInDf11 = maxXInYear

            minXInYear = x.min()
            if minXInYear < minXInDf11:
                minXInDf11 = minXInYear

            # C23
            #Mean
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "orange", label="mean = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
            #Median
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "orange", label="median = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(),2)))
            #Standard deviation
            axs[j].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

            meansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].mean()
            # print(meansTS1[i][j])
            mediansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].median()
            stdDevsTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].std()

            # Fitting
            axs[j].plot(np.sort(dataTS1[j].to_dataframe()[VarsToPlot[i]]), stats.norm.pdf(np.sort(dataTS1[j].to_dataframe()[VarsToPlot[i]]), meansTS1[i][j], stdDevsTS1[i][j]), color = "orange")
            dataTS1SansNans = dataTS1[j].to_dataframe()[VarsToPlot[i]]
            dataTS1SansNans = dataTS1SansNans[np.isfinite(dataTS1SansNans)]
            # results = anderson(np.sort(dataTS1SansNans), dist = 'norm')#, method = 'interpolate')
            results = anderson(np.sort(dataTS1SansNans), dist = 'norm', method = 'interpolate')
            statsResult1[i][j] = results.statistic
            print(results.statistic)
            # print(results.critical_values) # only if method is undefined
            # critValues[i][j] = results.critical_values
            pValues1[i][j] = results.pvalue

            if j < len(dataTS2):
                # axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs[j].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'green', label = "Cycle 24", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf21:
                    maxYInDf21 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf21:
                    maxXInDf21 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf21:
                    minXInDf21 = minXInYear

                # C24
                #Mean
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "green", label="mean = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "green", label="median = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs[j].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].std()

                # Fitting
                axs[j].plot(np.sort(dataTS2[j].to_dataframe()[VarsToPlot[i]]), stats.norm.pdf(np.sort(dataTS2[j].to_dataframe()[VarsToPlot[i]]), meansTS2[i][j], stdDevsTS2[i][j]), color = "green")
                dataTS2SansNans = dataTS2[j].to_dataframe()[VarsToPlot[i]]
                dataTS2SansNans = dataTS2SansNans[np.isfinite(dataTS2SansNans)]
                # results = anderson(np.sort(dataTS2SansNans), dist = 'norm')#, method = 'interpolate')
                results = anderson(np.sort(dataTS2SansNans), dist = 'norm', method = 'interpolate')
                statsResult2[i][j] = results.statistic
                pValues2[i][j] = results.pvalue

            if j < len(dataTS3):
                # axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs[j].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'blue', label = "Cycle 25", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf31:
                    maxYInDf31 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf31:
                    maxXInDf31 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf31:
                    minXInDf31 = minXInYear

                # C25
                #Mean
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "blue", label="mean = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "blue", label="median = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs[j].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].std()

                # Fitting
                axs[j].plot(np.sort(dataTS3[j].to_dataframe()[VarsToPlot[i]]), stats.norm.pdf(np.sort(dataTS3[j].to_dataframe()[VarsToPlot[i]]), meansTS3[i][j], stdDevsTS3[i][j]), color = "blue")
                dataTS3SansNans = dataTS3[j].to_dataframe()[VarsToPlot[i]]
                dataTS3SansNans = dataTS3SansNans[np.isfinite(dataTS3SansNans)]
                # results = anderson(np.sort(dataTS3SansNans), dist = 'norm')#, method = 'interpolate')
                results = anderson(np.sort(dataTS3SansNans), dist = 'norm', method = 'interpolate')
                statsResult3[i][j] = results.statistic
                pValues3[i][j] = results.pvalue

        # for j in range(len(NoOfYears)):
        #     if j < len(dataTS):
            axs[j].legend(loc = LegLoc, fontsize = Legend)
            axs[j].set_ylabel("Normalised frequency")

            # subplotLabels.append("Year" + str(j))
            axs[j].annotate("Year " + str(j + 1), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))
            # axs[j].annotate(str(StartAndEnd[j][0][0:4]), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

            # fig, axs = plt.subplot_mosaic([['a)', 'c)'], ['b)', 'c)'], ['d)', 'd)']], layout='constrained')
            # for label, ax in axs.items():
            #     ax.annotate(label, xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

        # maxYInDf1 = 0
        # for k in range(len(dataTS1)): # For each daterange
        #     maxInYear = max(dataTS1[k].to_dataframe()[VarsToPlot[i]])

        #     if maxInYear > maxYInDf1:
        #         maxYInDf1 = maxInYear

        # maxYInDf2 = 0
        # for l in range(len(dataTS2)): # For each daterange
        #     maxInYear = max(dataTS2[l].to_dataframe()[VarsToPlot[i]])
            
        #     if maxInYear > maxYInDf2:
        #         maxYInDf2 = maxInYear

        # maxYInDf3 = 0
        # for m in range(len(dataTS3)): # For each daterange
        #     maxInYear = max(dataTS3[m].to_dataframe()[VarsToPlot[i]])
            
        #     if maxInYear > maxYInDf3:
        #         maxYInDf3 = maxInYear
            
        
        fig2, axs2 = plt.subplots( (endSplit2-startSplit2) , 1, figsize=(FigSize[0], FigSize[1]*(endSplit2-startSplit2)/11), sharex=True)
        fig2.subplots_adjust(hspace=0)
        plt.suptitle("Wind Data, " + VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.9)
        fig2.supxlabel(VarsToPlot[i] + ", " + Units[i], y=0.085)
        plt.xscale(XAxisScale[i])
        # subplotLabels = []

        for j in range(startSplit2, endSplit2):

            # fig, axs = plt.subplots( (endSplit2-startSplit2) , 1, figsize=FigSize, sharex=True)
            # fig.subplots_adjust(hspace=0)
            # # plt.suptitle("Solar Wind "+ VarsToPlot[i] + ", " + Units[i], fontsize = SupTitle, y=0.89)
            # fig.supxlabel("Wind Data" + VarsToPlot[i] + ", " + Units[i], y=TitleLoc)
            # # subplotLabels = []

            axNo = j-startSplit2

            # axs[j].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'orange', label = "Cycle 23", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

            y, x, _ = axs2[axNo].hist(dataTS1[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'orange', label = "Cycle 23", density = True)
            maxInYear = y.max()
            if maxInYear > maxYInDf12:
                maxYInDf12 = maxInYear

            maxXInYear = x.max()
            if maxXInYear > maxXInDf12:
                maxXInDf12 = maxXInYear

            minXInYear = x.min()
            if minXInYear < minXInDf12:
                minXInDf12 = minXInYear

            # C23
            #Mean
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "orange", label="mean = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
            #Median
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "orange", label="median = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].median(),2)))
            #Standard deviation
            axs2[axNo].axvline(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS1[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

            meansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].mean()
            mediansTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].median()
            stdDevsTS1[i][j] = dataTS1[j].to_dataframe()[VarsToPlot[i]].std()

            # Fitting
            axs2[axNo].plot(np.sort(dataTS1[j].to_dataframe()[VarsToPlot[i]]), stats.norm.pdf(np.sort(dataTS1[j].to_dataframe()[VarsToPlot[i]]), meansTS1[i][j], stdDevsTS1[i][j]), color = "orange")
            dataTS1SansNans = dataTS1[j].to_dataframe()[VarsToPlot[i]]
            dataTS1SansNans = dataTS1SansNans[np.isfinite(dataTS1SansNans)]
            # results = anderson(np.sort(dataTS1SansNans), dist = 'norm')#, method = 'interpolate')
            results = anderson(np.sort(dataTS1SansNans), dist = 'norm', method = 'interpolate')
            statsResult1[i][j] = results.statistic
            pValues1[i][j] = results.pvalue

            
            if j < len(dataTS2):
                # axs[axNo].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'green', label = "Cycle 24", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs2[axNo].hist(dataTS2[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'green', label = "Cycle 24", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf22:
                    maxYInDf22 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf22:
                    maxXInDf22 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf22:
                    minXInDf22 = minXInYear

                # C24
                #Mean
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "green", label="mean = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "green", label="median = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs2[axNo].axvline(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS2[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS2[i][j] = dataTS2[j].to_dataframe()[VarsToPlot[i]].std()

                # Fitting
                axs2[axNo].plot(np.sort(dataTS2[j].to_dataframe()[VarsToPlot[i]]), stats.norm.pdf(np.sort(dataTS2[j].to_dataframe()[VarsToPlot[i]]), meansTS2[i][j], stdDevsTS2[i][j]), color = "green")
                dataTS2SansNans = dataTS2[j].to_dataframe()[VarsToPlot[i]]
                dataTS2SansNans = dataTS2SansNans[np.isfinite(dataTS2SansNans)]
                # results = anderson(np.sort(dataTS2SansNans), dist = 'norm')#, method = 'interpolate')
                results = anderson(np.sort(dataTS2SansNans), dist = 'norm', method = 'interpolate')
                statsResult2[i][j] = results.statistic
                pValues2[i][j] = results.pvalue

            if j < len(dataTS3):
                # axs[axNo].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bin, histtype='step', color = 'blue', label = "Cycle 25", density = True) #, label = "Wind " + str(StartAndEnd[j][0][0:10]) + " - " + str(StartAndEnd[j][1][0:10]), color=Colours[j], alpha = Alpha, density = True))

                y, x, _ = axs2[axNo].hist(dataTS3[j].to_dataframe()[VarsToPlot[i]], bins = Bins[i], histtype='step', color = 'blue', label = "Cycle 25", density = True)
                maxInYear = y.max()
                if maxInYear > maxYInDf32:
                    maxYInDf32 = maxInYear

                maxXInYear = x.max()
                if maxXInYear > maxXInDf32:
                    maxXInDf32 = maxXInYear

                minXInYear = x.min()
                if minXInYear < minXInDf32:
                    minXInDf32 = minXInYear

                # C25
                #Mean
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), linestyle = '--', color = "blue", label="mean = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].mean(), 2)))
                #Median
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), linestyle = ':', color = "blue", label="median = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].median(), 2)))
                #Standard deviation
                axs2[axNo].axvline(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), color = "grey", label="std dev = " + str(round(dataTS3[j].to_dataframe()[VarsToPlot[i]].std(), 2)), alpha = 0)

                meansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].mean()
                mediansTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].median()
                stdDevsTS3[i][j] = dataTS3[j].to_dataframe()[VarsToPlot[i]].std()

                # Fitting
                axs2[axNo].plot(np.sort(dataTS3[j].to_dataframe()[VarsToPlot[i]]), stats.norm.pdf(np.sort(dataTS3[j].to_dataframe()[VarsToPlot[i]]), meansTS3[i][j], stdDevsTS3[i][j]), color = "blue")
                dataTS3SansNans = dataTS3[j].to_dataframe()[VarsToPlot[i]]
                dataTS3SansNans = dataTS3SansNans[np.isfinite(dataTS3SansNans)]
                # results = anderson(np.sort(dataTS3SansNans), dist = 'norm')#, method = 'interpolate')
                results = anderson(np.sort(dataTS3SansNans), dist = 'norm', method = 'interpolate')
                statsResult3[i][j] = results.statistic
                pValues3[i][j] = results.pvalue
                
        # for j in range(len(NoOfYears)):
        #     if j < len(dataTS):
            axs2[axNo].legend(loc = LegLoc, fontsize = Legend)
            axs2[axNo].set_ylabel("Normalised frequency")

            # subplotLabels.append("Year" + str(j))
            axs2[axNo].annotate("Year " + str(j+1), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))
            # axs2[axNo].annotate(str(StartAndEnd[j][0][0:4]), xy=(0, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize', fontsize='medium', verticalalignment='top', fontfamily='serif', bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

        maxYsInDfs = [maxYInDf11, maxYInDf21, maxYInDf31, maxYInDf12, maxYInDf22, maxYInDf32]
        maxYOverall = max(maxYsInDfs)
        yScaleLim.append(maxYOverall)

        for j in range(startSplit1, endSplit1):
            # axs[j].set_ylim(0, maxYOverall)
            axs[j].set_ylim(0, maxYOverall*1.1) # adding 10% to y axis lim

        for j in range(startSplit2, endSplit2):
            axNo = j-startSplit2
            # axs2[axNo].set_ylim(0, maxYOverall)
            axs2[axNo].set_ylim(0, maxYOverall*1.1) # adding 10% to y axis lim

        maxXsInDfs = [maxXInDf11, maxXInDf21, maxXInDf31, maxXInDf12, maxXInDf22, maxXInDf32]
        maxXOverall = max(maxXsInDfs)
        XScaleMaxLim.append(maxXOverall)

        minXsInDfs = [minXInDf11, minXInDf21, minXInDf31, minXInDf12, minXInDf22, minXInDf32]
        minXOverall = min(minXsInDfs)
        XScaleMinLim.append(minXOverall)

        # # plt.xlim(min(LowerBounds) - SpaceLR[i], max(UpperBounds) + SpaceLR[i])
        # # plt.xlim(np.nanmin(LowerBounds) - SpaceLR[i], np.nanmax(UpperBounds) + SpaceLR[i])

        
        

        # # for m in range(len(dataTimeSerieses)):
        # #     if maxY == np.nan or maxY == np.inf:
        # #             print("Max is nan or inf.")
        #     # else:
        # for m in range(len(dataTimeSerieses)):
        #     # axs[m].set_ylim(0, maxY)
        #     axs[m].set_ylim(0, maxY+ 0.1*maxY)
        # # plt.tight_layout()
        # plt.yscale(YAxisScale[i])
        # fig.xscale(XAxisScale[i])
        # fig2.xscale(XAxisScale[i])
        
        # fig.savefig(PlotDirectory + StartAndEnd[startSplit1][0][0:10] + "->" + StartAndEnd[endSplit1][1][0:10] + VarsToPlot[i] + "FirstHalf_WindFittings.png")
        # fig2.savefig(PlotDirectory + StartAndEnd[startSplit2][0][0:10] + "->" + StartAndEnd[endSplit2][1][0:10] + VarsToPlot[i] + "SecondHalf_WindFittings.png")

        plt.show()
    return yScaleLim, XScaleMaxLim, XScaleMinLim, meansTS1, meansTS2, meansTS3, mediansTS1, mediansTS2, mediansTS3, stdDevsTS1, stdDevsTS2, stdDevsTS3, statsResult1, statsResult2, statsResult3, pValues1, pValues2, pValues3


# Plot CDF
def PlotCDF(dataTimeSerieses, VarsToPlot):
    # fig, axs = plt.subplots()#4,1, figsize=FigSize, sharex=True)
    # fig.subplots_adjust(hspace=0)
    for i in range(len(VarsToPlot)):
        for j in range(len(dataTimeSerieses)):
            sns.ecdfplot(dataTimeSerieses[j].to_dataframe()[VarsToPlot[i]], label="Wind " + str(StartAndEnd[j][0]) + " - " + str(StartAndEnd[j][1]), color=Colours[j])
        plt.xlabel(VarsToPlot[i] + ", " + Units[i])
        plt.ylabel('Cumulative probability')
        plt.ylim(yLim)
        plt.legend(fontsize=Legend)
        plt.grid(True)
        plt.savefig(PlotDirectory + StartAndEnd[0][0][0:10] + "->" + StartAndEnd[0][0][0:10] + VarsToPlot[i] + "_CDF_Wind.png") #  + dataTimeSerieses[0] + " - " + dataTimeSerieses[-1] 
        plt.show()


## Plot DQQs
# Calculate quantiles
def get_quantiles(
    x: pl.Series,
    y: pl.Series,
    size: int = 500, # 500 data points to calculate (default)
    p_range: Tuple[float, float] = (0.0001, 0.9999),) -> Tuple[NDArray, NDArray]: # p_range default set to values used in Tindale and Chapman 2017 to recreate their plots

    x_sorted = np.sort(x.drop_nulls().drop_nans().to_numpy()) # Sort data
    y_sorted = np.sort(y.drop_nulls().drop_nans().to_numpy()) # Sort data

    probabilities = np.linspace(*p_range, size) # make linspace of probabilities in range

    x_quantiles = np.quantile(x_sorted, probabilities) # calculate the quantiles
    y_quantiles = np.quantile(y_sorted, probabilities) # calculate the quantiles

    return x_quantiles, y_quantiles

## Simple FindKnee function without breakpoint finding
# def FindKnee(x, y):
#     direction, curve = find_shape(x, y)
#     kl = KneeLocator(x, y, curve=curve, direction=direction)
#     xOfKnee = kl.knee
#     yOfKnee = kl.knee_y
#     return xOfKnee, yOfKnee

# FindKnee function with breakpoint finding
def FindQuantOfKnee(x, y):
    direction, curve = find_shape(x, y)
    xlen = len(x)
    ylen = len(y)
    print(xlen, ylen)
    # kl = KneeLocator(x[int(xlen/2):-1], y[int(ylen/2):-1], curve=curve, direction=direction)
    kl = KneeLocator(x, y, curve=curve, direction=direction)
    xOfKnee = kl.knee
    yOfKnee = kl.knee_y
    print(xOfKnee, yOfKnee)
    
    XRes = stats.ecdf(x)#(xOfKnee)
    YRes = stats.ecdf(y)
    # print(Strawb.cdf.quantiles[4000])
    FindXidx = np.where(XRes.cdf.quantiles == xOfKnee)
    FindYidx = np.where(YRes.cdf.quantiles == yOfKnee)
    print(FindXidx, FindYidx)

    FindXidx = FindXidx[0][0]
    FindYidx = FindYidx[0][0]

    if FindXidx == FindYidx:
        ProbValue = XRes.cdf.probabilities[FindXidx]
        print(XRes.cdf.probabilities[FindXidx])
        print(YRes.cdf.probabilities[FindYidx])

    else:
        print("Indexes don't match")
        print(FindXidx, FindYidx)
        print(XRes.cdf.quantiles[FindXidx])
        print(YRes.cdf.quantiles[FindYidx])
        print("XRes -1")
        print(XRes.cdf.quantiles[FindXidx-1])
        print("YRes -1")
        print(YRes.cdf.quantiles[FindYidx-1])
        # ProbValue = XRes.cdf.probabilities[FindXidx]
        # ProbValue = YRes.cdf.probabilities[FindYidx]
        ProbValue = np.nan

    quantOfKnee = ProbValue
    return xOfKnee, yOfKnee, quantOfKnee

#Plot DQQ graphs with qs
def PlotDQQ(dataTimeSeriesToCompare, VarsToPlot, QuantilesSpecific):
    for i in range(len(VarsToPlot)): # do each parameter
        fig, ax  = plt.subplots(figsize=(5, 5))
        for j in range(len(dataTimeSeriesToCompare)): # do each comparison
            dataTimeSeriesPS1 = pl.Series(dataTimeSeriesToCompare[j][0].to_dataframe()[VarsToPlot[i]])
            dataTimeSeriesPS2 = pl.Series(dataTimeSeriesToCompare[j][1].to_dataframe()[VarsToPlot[i]])
            xQuant, yQuant = get_quantiles(dataTimeSeriesPS1, dataTimeSeriesPS2, size = 5000, p_range=(0.0001, 0.9999))
            plt.scatter(xQuant, yQuant, color=Color[j], s=2, label = PlotLabels[j])

            xOfKnee, yOfKnee, quantOfKnee = FindQuantOfKnee(xQuant, yQuant)

            if np.isnan(quantOfKnee):
                print("---------------------------")
                print("Error finding breakpoints q")
                print("---------------------------")
            else:
                plt.plot(xOfKnee, yOfKnee, "o", color="black", label = "q = " + str(quantOfKnee))
            # plt.plot(xOfKnee, yOfKnee, "o", color="black")
        
        x = np.linspace(-100, 900, 2)
        y = x
        plt.plot(x, y, color="grey")
        plt.title("T&C 2017 Fig 4aiii)")
        plt.legend(loc="lower right", fontsize = Legend)
        plt.xlabel(VarsToPlot[i] + r", $km/s$, solar min.")
        plt.ylabel(VarsToPlot[i] + r", $km/s$, solar max.")
        plt.xlim(Lims[i]) #min(xQuantC23)-np.absolute(min(xQuantC23)/2) , max(xQuantC23)+np.absolute(max(xQuantC23)/2))
        plt.ylim(Lims[i]) #min(xQuantC23)-np.absolute(min(xQuantC23)/2), max(xQuantC23)+np.absolute(max(xQuantC23)/2))
        plt.savefig(PlotDirectory + "DQQ_Wind.png")

        plt.show()